##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [ ]:
# place your code here
#%pip install pinecone

From 'pinecone' library import Pinecone, ServerlessSpec.

In [1]:
# place your code here
from pinecone import Pinecone, ServerlessSpec

On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [2]:
# place your code here
import os
from dotenv import load_dotenv

load_dotenv()

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [3]:
index_name = "ktu-semantic-search"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [5]:
sentences_list = [
    "Kaunas University of Technology provides several courses for bachelor's and master's students in the field of computer science.",
    "Students at KTU can choose different study programmes related to software engineering, artificial intelligence, and information technology.",
    "The university offers opportunities for students to participate in research projects and gain practical experience during their studies.",
    "Computer science students learn programming languages, algorithms, databases, and software development as part of their academic education.",
    "Many international students choose Kaunas University of Technology because of its English taught programmes and multicultural learning environment.",
    "KTU students can take part in internships and collaborate with companies to gain experience before graduating from university.",
    "The university provides bachelor's and master's degree programmes in engineering, business, technology, and other academic disciplines.",
    "Students studying computer science often work on practical projects involving artificial intelligence, machine learning, and software development."
]

In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

embeddings = model.encode(sentences_list)

vectors = [
    {
        "id": str(i),
        "values": embedding.tolist(),
        "metadata": {"text": sentence}
    }
    for i, (sentence, embedding) in enumerate(zip(sentences_list, embeddings))
]

index.upsert(vectors=vectors)

print("Vectors uploaded:", len(vectors))

c:\Users\ali.a.mehdi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2384.82it/s]


Vectors uploaded: 8


Print search results.

In [ ]:
# place your code here
query = "I want to study artificial intelligence and gain practical experience at KTU."

query_embedding = model.encode(query).tolist()

results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

for match in results["matches"]:
    print("Score:", round(match["score"], 4))
    print("Text:", match["metadata"]["text"])
    print()

Score: 0.7276
Text: Students at KTU can choose different study programmes related to software engineering, artificial intelligence, and information technology.

Score: 0.5042
Text: KTU students can take part in internships and collaborate with companies to gain experience before graduating from university.

Score: 0.4456
Text: Kaunas University of Technology provides several courses for bachelor's and master's students in the field of computer science.

Score: 0.4379
Text: Students studying computer science often work on practical projects involving artificial intelligence, machine learning, and software development.

Score: 0.4212
Text: Many international students choose Kaunas University of Technology because of its English taught programmes and multicultural learning environment.



# Observation
Well in this part of the experiment we used pinecone. By using this library we got the most relevant result with a similarity score of 0.7276 while matching the question about studying AI and choosing related programmes at KTU and internship related sentence came second with 0.5042. Whereas the remaining results had lower scores. After applying those i got to know how to store embeddings in a cloud vector database along with metadata so the original text can be retrieved alongside the similarity score. This showed me how Pinecone can handle semantic searches without manually comparing vectors making it useful for the build of retrieval part of a RAG application.